# 🌿 EcoTrack-AI: Emission Data Analysis
**Exploratory data analysis on carbon footprint tracking data.**  
Covers trend analysis, anomaly detection review, user behaviour patterns, and month-over-month insights using pandas and numpy.

---


## 1. Setup & Synthetic Data Generation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#f9fbf7',
    'axes.facecolor': '#f0f4ee',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
GREEN = '#2d6a4f'
ORANGE = '#f4a261'
RED = '#e76f51'
BLUE = '#457b9d'

print("Libraries loaded.")

## 2. Generate Realistic Emission Dataset

In [ ]:
n_users = 120
n_days = 180  # 6 months: Jan–Jun 2025

date_range = pd.date_range(start='2025-01-01', periods=n_days, freq='D')
users = [f'user_{i:03d}' for i in range(1, n_users + 1)]

# User profiles: heavy, moderate, eco-conscious
profiles = np.random.choice(['heavy', 'moderate', 'eco'], size=n_users, p=[0.25, 0.50, 0.25])
base_emission = {'heavy': 28, 'moderate': 16, 'eco': 8}

records = []
for user, profile in zip(users, profiles):
    base = base_emission[profile]
    for date in date_range:
        # Weekly pattern: higher on weekdays
        weekday_bump = 1.15 if date.weekday() < 5 else 0.80
        # Seasonal: slight drop in spring (Apr–May)
        seasonal = 0.88 if date.month in [4, 5] else 1.0
        # Gradual improvement trend
        trend = 1 - (date.dayofyear / n_days) * 0.12
        noise = np.random.normal(1.0, 0.12)
        emission = base * weekday_bump * seasonal * trend * noise
        # Inject anomalies (~3%)
        is_anomaly = np.random.rand() < 0.03
        if is_anomaly:
            emission *= np.random.uniform(2.5, 4.0)
        records.append({
            'date': date,
            'user_id': user,
            'profile': profile,
            'emission_kg': round(max(emission, 0.5), 3),
            'is_anomaly': is_anomaly,
            'month': date.strftime('%b'),
            'month_num': date.month,
            'week': date.isocalendar()[1],
            'weekday': date.day_name(),
        })

df = pd.DataFrame(records)
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df.date.min().date()} → {df.date.max().date()}")
print(f"\nEmission stats (kg CO2):")
print(df['emission_kg'].describe().round(2))

## 3. Month-over-Month Emission Trends

In [ ]:
monthly = df.groupby(['month_num', 'month'])['emission_kg'].agg(
    total='sum', mean='mean', median='median', users='nunique'
).reset_index().sort_values('month_num')

monthly['mom_change_pct'] = monthly['mean'].pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean daily emission per month
ax = axes[0]
bars = ax.bar(monthly['month'], monthly['mean'], color=GREEN, alpha=0.85, width=0.6)
ax.set_title('Average Daily Emission by Month', fontweight='bold', pad=12)
ax.set_ylabel('Avg Emission (kg CO2)')
ax.set_xlabel('Month')
for bar, val in zip(bars, monthly['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.1f}', ha='center', va='bottom', fontsize=9, color='#333')

# MoM % change
ax2 = axes[1]
colors = [GREEN if x <= 0 else RED for x in monthly['mom_change_pct'].fillna(0)]
ax2.bar(monthly['month'], monthly['mom_change_pct'].fillna(0), color=colors, alpha=0.85, width=0.6)
ax2.axhline(0, color='#333', linewidth=0.8, linestyle='--')
ax2.set_title('Month-over-Month Change (%)', fontweight='bold', pad=12)
ax2.set_ylabel('Change (%)')
ax2.set_xlabel('Month')

plt.suptitle('EcoTrack-AI: Monthly Emission Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('monthly_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nMonth-over-Month summary:")
print(monthly[['month', 'mean', 'median', 'mom_change_pct']].to_string(index=False))

## 4. Emission Distribution by User Profile

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = {'heavy': RED, 'moderate': ORANGE, 'eco': GREEN}

# KDE distribution
ax = axes[0]
for profile, color in palette.items():
    subset = df[df.profile == profile]['emission_kg']
    subset[subset < 80].plot.kde(ax=ax, label=profile.title(), color=color, linewidth=2)
ax.set_title('Emission Distribution by Profile', fontweight='bold', pad=12)
ax.set_xlabel('Emission (kg CO2)')
ax.set_ylabel('Density')
ax.legend()
ax.set_xlim(0, 80)

# Boxplot
ax2 = axes[1]
df_clean = df[df.emission_kg < df.emission_kg.quantile(0.97)]  # exclude top anomalies for viz
data_by_profile = [df_clean[df_clean.profile == p]['emission_kg'].values for p in ['eco', 'moderate', 'heavy']]
bp = ax2.boxplot(data_by_profile, patch_artist=True, notch=True,
                 labels=['Eco', 'Moderate', 'Heavy'])
for patch, color in zip(bp['boxes'], [GREEN, ORANGE, RED]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax2.set_title('Emission Range by Profile (Anomalies Excluded)', fontweight='bold', pad=12)
ax2.set_ylabel('Emission (kg CO2)')

plt.tight_layout()
plt.savefig('profile_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

profile_stats = df.groupby('profile')['emission_kg'].agg(['mean','median','std']).round(2)
print("\nProfile Statistics:")
print(profile_stats)

## 5. Anomaly Detection Review

In [ ]:
# Z-score based detection on top of injected anomalies
daily_avg = df.groupby(['date', 'user_id'])['emission_kg'].sum().reset_index()
daily_avg['z_score'] = np.abs(stats.zscore(daily_avg['emission_kg']))
daily_avg['detected_anomaly'] = daily_avg['z_score'] > 2.8

# Aggregate anomalies by week
weekly_anomalies = daily_avg.groupby(daily_avg['date'].dt.isocalendar().week)['detected_anomaly'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Weekly anomaly count
ax = axes[0]
ax.bar(weekly_anomalies.index, weekly_anomalies.values, color=RED, alpha=0.75, width=0.7)
ax.set_title('Detected Anomalies per Week', fontweight='bold', pad=12)
ax.set_xlabel('Week Number')
ax.set_ylabel('Anomaly Count')

# Scatter: emission vs z-score
ax2 = axes[1]
normal = daily_avg[~daily_avg['detected_anomaly']]
anomalous = daily_avg[daily_avg['detected_anomaly']]
ax2.scatter(normal['emission_kg'], normal['z_score'], alpha=0.3, color=GREEN, s=8, label='Normal')
ax2.scatter(anomalous['emission_kg'], anomalous['z_score'], alpha=0.8, color=RED, s=20, label='Anomaly')
ax2.axhline(2.8, color='#333', linestyle='--', linewidth=0.9, label='Threshold (z=2.8)')
ax2.set_title('Emission vs Z-Score', fontweight='bold', pad=12)
ax2.set_xlabel('Emission (kg CO2)')
ax2.set_ylabel('Z-Score')
ax2.legend()

plt.tight_layout()
plt.savefig('anomaly_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

total = len(daily_avg)
detected = daily_avg['detected_anomaly'].sum()
print(f"Total records: {total:,}")
print(f"Anomalies detected: {detected} ({detected/total*100:.2f}%)")

## 6. Weekday vs Weekend Emission Pattern

In [ ]:
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekday_stats = df.groupby('weekday')['emission_kg'].mean().reindex(order)

fig, ax = plt.subplots(figsize=(10, 4))
bar_colors = [BLUE if d not in ['Saturday','Sunday'] else GREEN for d in order]
bars = ax.bar(weekday_stats.index, weekday_stats.values, color=bar_colors, alpha=0.85, width=0.6)
ax.set_title('Average Emission by Day of Week', fontweight='bold', pad=12)
ax.set_ylabel('Avg Emission (kg CO2)')
for bar, val in zip(bars, weekday_stats.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}', ha='center', fontsize=8.5, color='#333')

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=BLUE, label='Weekday'), Patch(color=GREEN, label='Weekend')])
plt.tight_layout()
plt.savefig('weekday_pattern.png', dpi=150, bbox_inches='tight')
plt.show()

weekday_mean = df[df['weekday'].isin(['Monday','Tuesday','Wednesday','Thursday','Friday'])]['emission_kg'].mean()
weekend_mean = df[df['weekday'].isin(['Saturday','Sunday'])]['emission_kg'].mean()
print(f"Weekday avg: {weekday_mean:.2f} kg CO2")
print(f"Weekend avg: {weekend_mean:.2f} kg CO2")
print(f"Weekdays emit {((weekday_mean - weekend_mean)/weekend_mean)*100:.1f}% more than weekends")

## 7. Key Insights

| Insight | Finding |
|---|---|
| **Overall trend** | Emissions drop ~12% over 6 months as users adopt eco-habits |
| **Seasonal dip** | April–May shows lowest average emissions (spring effect) |
| **Heavy vs Eco** | Heavy users emit ~3.5x more than eco-conscious users on average |
| **Anomaly rate** | ~3% of daily readings flagged as anomalies via z-score threshold |
| **Weekday pattern** | Weekday emissions run ~15% higher than weekends |
| **Best month** | May consistently shows lowest average daily emissions |

---
*Dataset: 120 users, 180 days (Jan–Jun 2025), 21,600 records. Synthetic data generated to mirror realistic carbon footprint tracking patterns.*
